<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/10_ML_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Foundations — Airtel Customer Churn
**Question:** which customers will cancel next month, so retention can call them first?


## 0. Load the data

In [ ]:
!gdown 1EaY4IGoRpcxv1nIidrHbkduOmWVplkh2

Downloading...
From: https://drive.google.com/uc?id=1EaY4IGoRpcxv1nIidrHbkduOmWVplkh2
To: /content/Telco-Customer-Churn.csv
100% 978k/978k [00:00<00:00, 107MB/s]


In [ ]:
import pandas as pd

df = pd.read_csv("Telco-Customer-Churn.csv")
df.shape

(7043, 21)

In [ ]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
df.isna().sum().sum()

np.int64(0)

In [ ]:
df[df["TotalCharges"] == " "].shape[0]

11

In [ ]:
# TotalCharges is stored as text with 11 blanks -- quick fix (full cleaning is next class)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df.shape

(7032, 21)

In [ ]:
df["Churn"].value_counts(normalize=True) # what this code does ?

,proportion
Churn,
No,0.734215
Yes,0.265785


Problem Statement:

- How much data (GB) will a customer use next month?
- Which plan will a customer pick: Basic, Plus or Unlimited?
- Group 7 million customers into segments for marketing

## 1. Rules vs Learning

In [ ]:
df[['Contract', 'Churn']].head()

,Contract,Churn
0,Month-to-month,No
1,One year,No
2,Month-to-month,Yes
3,One year,No
4,Month-to-month,Yes


In [ ]:
df.groupby("Contract")["Churn"].value_counts(normalize=True)

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887228
                Yes      0.112772
Two year        No       0.971513
                Yes      0.028487
Name: proportion, dtype: float64

In [ ]:
# A human-written rule: "month-to-month customers will churn"
rule_says_churn = df["Contract"] == "Month-to-month"
actually_churned = df["Churn"] == "Yes"

## 2. Features (X) and Target (y)

In [ ]:
X = df[["tenure", "MonthlyCharges", "TotalCharges"]]
y = df["Churn"]

In [ ]:
X.head()

,tenure,MonthlyCharges,TotalCharges
0,1,29.85,29.85
1,34,56.95,1889.50
2,2,53.85,108.15
3,45,42.30,1840.75
4,2,70.70,151.65


In [ ]:
y.head()

,Churn
0,No
1,No
2,Yes
3,No
4,Yes


## 3. Train / Test Split

In [ ]:
X.shape

(7032, 3)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

len(X_train), len(X_test)

(5625, 1407)

In [ ]:
y_test.value_counts(normalize=True)

,proportion
Churn,
No,0.734186
Yes,0.265814


## 4. The Pattern: create → fit → predict → score

In [ ]:
# Baseline: always predict the most common answer ("Not churn")
from sklearn.dummy import DummyClassifier

# 1. create a model
baseline = DummyClassifier()

# 2. fit train a model
baseline.fit(X_train, y_train)

# 3. predict
y_predict = baseline.predict(X_test)

# 4. accuracy/evaluate model (on test data)
score = baseline.score(X_test, y_test)

print("Baseline/Dummy model accuracy:", score)

Baseline/Dummy model accuracy: 0.7341862117981521


In [ ]:
from sklearn.linear_model import LogisticRegression # it's a classification model.

model = LogisticRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = model.score(X_test, y_test)

print("Logistic Regression model accuracy:", accuracy)

Logistic Regression model accuracy: 0.7796730632551528


In [ ]:
# Accuracy hides the real question: how many churners did we catch?
predictions = model.predict(X_test)
pd.crosstab(y_test, predictions)

col_0,No,Yes
Churn,,
No,937,96
Yes,214,160


## 5. Same Pattern, Any Algorithm

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

model = DecisionTreeClassifier(max_depth=5) # KNeighborsClassifier(n_neighbors=7)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = model.score(X_test, y_test)
print("kNN model accuracy:", accuracy)

predictions = model.predict(X_test)
pd.crosstab(y_test, predictions)

kNN model accuracy: 0.7818052594171997


col_0,No,Yes
Churn,,
No,945,88
Yes,219,155


## 7. Predicting the Future

In [ ]:
new_customers = pd.DataFrame({
    "tenure":         [65],
    "MonthlyCharges": [195],
    "TotalCharges":   [1900],
})

new_customers

,tenure,MonthlyCharges,TotalCharges
0,65,195,1900


In [ ]:
model.predict(new_customers)

array(['No'], dtype=object)

In [ ]:
# Probability of "Yes"
model.predict_proba(new_customers)

array([[0.72571429, 0.27428571]])

## 8. Regression: Same Pattern, Different Target

In [ ]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
X_reg = df[["tenure", "MonthlyCharges"]]
y_reg = df["TotalCharges"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [ ]:
Xr_train.head()

,tenure,MonthlyCharges
6030,43,49.05
3410,3,53.40
5483,55,77.75
5524,45,54.65
6337,55,100.90


In [ ]:
yr_train.head()

,TotalCharges
6030,2076.20
3410,188.70
5483,4458.15
5524,2553.70
6337,5448.60


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

baseline_reg = DummyRegressor()          # always predicts the average
baseline_reg.fit(Xr_train, yr_train)
mean_absolute_error(yr_test, baseline_reg.predict(Xr_test))

1907.318400909737

In [ ]:
reg_model = LinearRegression()
reg_model.fit(Xr_train, yr_train)
reg_predictions = reg_model.predict(Xr_test)
mean_absolute_error(yr_test, reg_predictions)

571.0893527607234

In [ ]:
pd.DataFrame({"actual": yr_test[:5].values, "predicted": reg_predictions[:5].round()})

,actual,predicted
0,1501.75,2731.0
1,465.85,-24.0
2,1359.00,2353.0
3,2030.75,2236.0
4,161.45,-704.0


## Recap — always same pattern
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = AnyModel()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
model.score(X_test, y_test)   #  compare against a baseline
```